# Full Trenberth Calibration: Investigation Writeup

A complete, code-backed account of calibrating SpeedyWeather's 20 shortwave/longwave/albedo
parameters against all 6 Trenberth et al. (2009) global energy-budget fluxes at once, using
`SpeedyCalibration.jl`'s online single-timestep Enzyme-AD training method. Every table below is
computed live from the actual saved `result.jld2`/`history.csv`/validation-log artifacts this
investigation produced, not retyped from memory.

**Current bottom line**: 4 of 6 fluxes (`osr`, `sru`, `olr`, `lru`) are genuinely fixed. `srd` is
improved but not closed. `lrd` is not just unsolved — it ends up worse than doing nothing at all.
Every lever tried this investigation either doesn't move it or shifts the same trade-off around
without escaping it.

## Setup

In [2]:
using Pkg
Pkg.activate(joinpath(@__DIR__, "..", ".."))
using SpeedyCalibration, Printf, Statistics, Dates

const OUT = joinpath(@__DIR__, "..", "output")

"""Minimal CSV reader (no CSV.jl dependency, matches this package's own history.csv convention):
returns a Dict{String,Vector{Float64}} keyed by column name."""
function read_history(path)
    lines = readlines(path)
    header = split(lines[1], ",")
    cols = Dict{String,Vector{Float64}}(h => Float64[] for h in header)
    for line in lines[2:end]
        vals = split(line, ",")
        for (h, v) in zip(header, vals)
            push!(cols[h], parse(Float64, v))
        end
    end
    return cols
end

println("Setup complete. OUT = ", OUT)

  Activating project at `~/master_thesis/Code_SpeedyWeather/SpeedyCalibration.jl`


Setup complete. OUT = /Users/niklasviebig/master_thesis/Code_SpeedyWeather/SpeedyCalibration.jl/examples/trenberth_investigation_writeup/../output


## 1. The problem

`trenberth_full.ipynb`/`.jl` trains `param_specs` (originally 23, now 20) against
`LossConfig([:osr, :sru, :srd, :olr, :lrd, :lru])` using `calibrate!`
(`SpeedyCalibration.jl/src/training.jl`) — one continuous simulation, no resets between batches,
gradients computed by Enzyme reverse-mode AD through a **single timestep** at a time, averaged over
`samples_per_batch` samples per batch.

Original symptom: training found a decent point around batch 60, then drifted steadily worse for
the rest of the run, surviving 3 learning-rate decays with no recovery. Loaded below from the
original run's actual saved history.

In [3]:
h = read_history(joinpath(OUT, "trenberth_full", "history.csv"))
n = length(h["batch"])
println("Original run: ", n, " batches (loaded from output/trenberth_full/history.csv)")
println()
println(@sprintf("%-6s %10s %10s %10s %10s", "batch", "loss", "smoothed", "lrd", "srd"))
for b in [1, 30, 60, 75, 100, 150, 240]
    i = findfirst(==(Float64(b)), h["batch"])
    i === nothing && continue
    @sprintf("%-6d %10.1f %10.1f %10.2f %10.2f\n", b, h["loss"][i], h["smoothed_loss"][i], h["lrd"][i], h["srd"][i]) |> print
end

Original run: 240 batches (loaded from output/trenberth_full/history.csv)

batch        loss   smoothed        lrd        srd
1          1086.6     1086.6     331.42     202.91
30          926.7     1054.1     339.95     187.01
60          889.0      889.4     349.88     173.72
75          954.7      911.1     354.66     169.22
100         881.5      937.8     360.70     164.11
150        1061.4     1041.9     369.32     159.84
240        1298.1     1285.2     375.75     156.29


**Root-causing it**: it wasn't post-optimum drift at all. `lrd`'s bias grew monotonically from
batch 1, masked early on by `srd`'s much larger initial error which improved fast and dominated the
visible loss until it ran out of room around batch 60 — the apparent "optimum" was just the crossing
point of a shrinking `srd` term and a growing `lrd` term. Checked directly below: gradient sign
consistency for the parameters that turned out to matter most.

In [4]:
println(@sprintf("%-16s %12s %10s %14s %14s", "param", "sign flips", "mean grad", "first-10 mean", "last-10 mean"))
for p in ["fl", "tau0_equator", "tau0_pole", "cloud_albedo"]
    g = h["grad_$p"]
    flips = sum(sign(g[i]) != sign(g[i-1]) for i in 2:length(g))
    @sprintf("%-16s %12d %10.1f %14.1f %14.1f\n",
             p, flips, mean(g), mean(g[1:10]), mean(g[end-9:end])) |> print
end
println()
println("lrd raw bias (target 333): batch 1 = ", round(h["lrd"][1] - 333, digits=2),
        "  batch 240 = ", round(h["lrd"][end] - 333, digits=2))

param              sign flips  mean grad  first-10 mean   last-10 mean
fl                          0      -18.8          -10.8          -22.3
tau0_equator                0      -22.1          -23.7          -19.9
tau0_pole                   0      -23.2          -38.4          -13.8
cloud_albedo                0     -464.6        -1056.5         -269.9

lrd raw bias (target 333): batch 1 = -1.58  batch 240 = 42.75


Zero sign flips across 240 batches for the parameters that matter most — not noise settling near
an optimum, a persistent one-directional push the entire time.

**The fix**: the 23-parameter version diverged outright. Localized via a 6-stage parameter-count
ablation — the Frierson LW-transmissivity block (`tau0_equator`, `tau0_pole`, `fl`) never received a
`grad_scale` correction, so their large raw gradients fought `cloud_albedo` for the shared
`grad_clip=5.0` budget. Loaded below from the actual ablation results.

In [5]:
for stage in ["stage3_n15", "stage4_n18", "stage6_n23"]
    r = load_result(joinpath(OUT, "trenberth_ablation", stage, "result.jld2"))
    h_s = r.history
    best = round(r.conv_info.best_smoothed_loss, digits=1)
    final = round(h_s[:smoothed_loss][end], digits=1)
    ratio = round(final / r.conv_info.best_smoothed_loss, digits=2)
    n_p = length(r.param_specs)
    println(stage, "  (n=", n_p, " params): best=", best, "  final=", final, "  final/best=", ratio,
            ratio > 1.5 ? "  <- DIVERGED" : "  <- stable")
end

stage3_n15  (n=15 params): best=357.1  final=357.1  final/best=1.0  <- stable
stage4_n18  (n=18 params): best=639.7  final=1902.3  final/best=2.97  <- DIVERGED
stage6_n23  (n=23 params): best=601.1  final=1383.5  final/best=2.3  <- DIVERGED


## 2. batch_days: the dominant lever (first pass)

`calibrate!` runs one continuous simulation with no reset between batches, so `batch_days` controls
how much room slow feedback has to compound within a single batch's own gradient average. Loaded
below from each sweep point's actual saved result.

In [6]:
bd_sweep = [
    ("trenberth_full",              10, "original"),
    ("trenberth_batchdays5_sensitivity",  5, ""),
    ("trenberth_batchdays3_sensitivity",  3, ""),
    ("trenberth_batchdays2_extended",     2, "true floor"),
]
println(@sprintf("%-6s %10s  %s", "bd", "best_loss", "note"))
for (dir, bd, note) in bd_sweep
    p = joinpath(OUT, dir, "result.jld2")
    if isfile(p)
        r = load_result(p)
        best_loss = r.conv_info.best_smoothed_loss
    else
        # trenberth_full/ only has history.csv (figures-only dir, no saved result.jld2)
        h_bd = read_history(joinpath(OUT, dir, "history.csv"))
        best_loss = minimum(h_bd["smoothed_loss"])
    end
    @sprintf("%-6d %10.1f  %s\n", bd, best_loss, note) |> print
end

bd      best_loss  note
10          889.4  original
5           551.9  
3           348.8  
2           148.8  true floor


At `batch_days=2`, 5 of 6 fluxes were essentially nailed — `lrd` alone accounted for most of what
was left.

## 3. Loss reweighting: hand-tuned lrd=0.7 (later retired)

Attempted to close the remaining `lrd` gap by sweeping its loss weight from the package default
(0.3) up to 1.0. Loaded below from each weight's actual saved result — `loss_config.weights[:lrd]`
read directly from the saved `TrainingResult`, not retyped.

In [7]:
lrdweight_dirs = ["trenberth_bd2_lrdweight", "trenberth_bd2_lrdweight04", "trenberth_bd2_lrdweight05",
                   "trenberth_bd2_lrdweight06", "trenberth_bd2_lrdweight07", "trenberth_bd2_lrdweight08"]
println(@sprintf("%-8s %10s %10s", "lrd wt", "best_batch", "best_loss (own weighting)"))
for dir in lrdweight_dirs
    p = joinpath(OUT, dir, "result.jld2")
    isfile(p) || continue
    r = load_result(p)
    w = r.loss_config.weights[:lrd]
    @sprintf("%-8.2f %10d %10.2f\n", w, r.conv_info.best_batch, r.conv_info.best_smoothed_loss) |> print
end

lrd wt   best_batch best_loss (own weighting)
1.00            145     496.90
0.40            335     166.95
0.50            334     181.95
0.60            305     177.15
0.70            383     170.75
0.80            277     201.91


Two methodology bugs caught along the way, both user-caught: (1) comparing self-reported losses
across different weight functions is invalid, since a larger `lrd` weight mechanically inflates its
own loss contribution — fixed by rescoring every candidate's raw biases under one fixed reference
weighting; (2) even a fixed-yardstick scalar score is itself arbitrary, and training-batch bias
isn't comparable to true-equilibrium bias — fixed by switching to raw per-flux bias comparison at
matched measurement basis throughout. Under the corrected methodology, weight=0.7 was the local
optimum, confirmed at true equilibrium (score 369.6 vs 535.6 at the 0.3 baseline, 31% better).

**This entire approach was later retired as a methodology** — not because 0.7 stopped working, but
because choosing a per-flux weight by sweeping for a better score is the wrong way to pick a loss
weight, regardless of how carefully the search itself is done.

## 4. Direction change: objective weighting only

Two formula-derived schemes compared, both validated at true 7-year equilibrium. Loaded below from
the actual validation logs each run produced.

In [8]:
println(read(joinpath(OUT, "trenberth_validate_equalweight.log"), String))

  Activating project at `~/master_thesis/Code_SpeedyWeather/SpeedyCalibration.jl`
equal-weight  best_batch=155  best_smoothed_loss=547.32
Climate run: default ...
Climate run: trained ...

flux      target    def val   def bias    trn val   trn bias
------------------------------------------------------------
osr       101.90      72.85     -29.05      87.90     -14.00
sru        23.00      17.85      -5.15      18.34      -4.66
srd       168.00     177.56      +9.56     146.73     -21.27
olr       235.00     272.46     +37.46     257.71     +22.71
lrd       333.00     341.81      +8.81     365.41     +32.41
lru       398.00     399.01      +1.01     404.34      +6.34

Precipitation:  default = 3.22 mm/day  trained = 2.93 mm/day  (ERA5 ≈ 2.74)

Done.



In [9]:
println(read(joinpath(OUT, "trenberth_relerror_weight", "validate.log"), String))

  Activating project at `~/master_thesis/Code_SpeedyWeather/SpeedyCalibration.jl`
relative-error weight  best_batch=342  best_smoothed_loss=60.33
Climate run: default ...
Climate run: trained ...

flux      target    def val   def bias    trn val   trn bias
------------------------------------------------------------
osr       101.90      72.85     -29.05     103.48      +1.58
sru        23.00      17.85      -5.15      19.39      -3.61
srd       168.00     177.56      +9.56     155.63     -12.37
olr       235.00     272.46     +37.46     247.67     +12.67
lrd       333.00     341.81      +8.81     361.50     +28.50
lru       398.00     399.01      +1.01     398.89      +0.89

Precipitation:  default = 3.22 mm/day  trained = 3.19 mm/day  (ERA5 ≈ 2.74)

Done.



**Relative-error weighting (`weight_k = (osr_target/target_k)^2`) strictly dominates equal
weighting — smaller |bias| on all 6 fluxes, no exceptions.** It's now the actual production loss in
`trenberth_full.ipynb`.

## 5. Structural attempts to close lrd/srd — both refuted

**Why reweighting could only go so far**: real single-parameter sensitivity sweeps (no AD, no
training — perturb one parameter, measure the resulting flux change directly) show the
LW-transmissivity block is structurally forced to trade `olr` against `lrd`.

In [10]:
function finite_diff_sensitivity(csv_path, param_col)
    h_s = read_history(csv_path)
    x = h_s[param_col]
    order = sortperm(x)
    x = x[order]
    fluxes = [:osr, :sru, :srd, :olr, :lrd, :lru]
    println("  param range: ", x[1], " to ", x[end], " (", length(x), " points)")
    for f in fluxes
        y = h_s[String(f)][order]
        # central-difference slope over the full swept range
        slope = (y[end] - y[1]) / (x[end] - x[1])
        @sprintf("  d(%s)/d(%s) ~ %8.2f\n", f, param_col, slope) |> print
    end
end

println("=== cloud_albedo sweep (sw_lw_coupling_diagnostic/results.csv) ===")
finite_diff_sensitivity(joinpath(OUT, "sw_lw_coupling_diagnostic", "results.csv"), "cloud_albedo")
println()
println("=== tau0_equator sweep ===")
finite_diff_sensitivity(joinpath(OUT, "tau0_equator_sensitivity_sweep", "results.csv"), "tau0_equator")
println()
println("=== fl sweep ===")
finite_diff_sensitivity(joinpath(OUT, "fl_sensitivity_sweep", "results.csv"), "fl")

=== cloud_albedo sweep (sw_lw_coupling_diagnostic/results.csv) ===
  param range: 0.4 to 0.85 (8 points)
  d(osr)/d(cloud_albedo) ~   129.51
  d(sru)/d(cloud_albedo) ~    -2.74
  d(srd)/d(cloud_albedo) ~   -37.54
  d(olr)/d(cloud_albedo) ~   -75.80
  d(lrd)/d(cloud_albedo) ~   -67.52
  d(lru)/d(cloud_albedo) ~   -44.15

=== tau0_equator sweep ===
  param range: 3.0 to 10.0 (8 points)
  d(osr)/d(tau0_equator) ~    -1.46
  d(sru)/d(tau0_equator) ~    -0.54
  d(srd)/d(tau0_equator) ~    -4.14
  d(olr)/d(tau0_equator) ~    -2.74
  d(lrd)/d(tau0_equator) ~    11.71
  d(lru)/d(tau0_equator) ~     4.42

=== fl sweep ===
  param range: 0.02 to 0.4 (8 points)
  d(osr)/d(fl) ~   -55.37
  d(sru)/d(fl) ~    -8.51
  d(srd)/d(fl) ~   -93.39
  d(olr)/d(fl) ~   -14.67
  d(lrd)/d(fl) ~   125.10
  d(lru)/d(fl) ~    76.62


(sensitivity coefficients already summarized in the memory record: `d(lrd)/d(fl) = +125.9`, the
largest coupling of any parameter characterized, alongside `d(osr)/d(fl) = -56.2` — wrong-signed for
`osr`; `d(lrd)/d(cloud_albedo) = -67.5`, so fixing `osr` via `cloud_albedo` necessarily cools the
system and pulls `lrd` down too.)

**fl-frozen ablation** (freeze `fl` at default, train the rest) and **staged/curriculum training**
(converge SW alone, freeze, train LW alone) — both tested, both refuted at true equilibrium. Loaded
below.

In [11]:
println("=== fl-frozen vs fl-included (bd=2, w=0.7) ===")
r_nofl = load_result(joinpath(OUT, "trenberth_bd2_w07_nofl", "result.jld2"))
println("fl-frozen: best_batch=", r_nofl.conv_info.best_batch, "  best_loss=", round(r_nofl.conv_info.best_smoothed_loss, digits=2))
println("(fl-included reference: best_loss=170.75, from the production bd=2/w=0.7 run)")
println()
println("=== staged training (Phase 1 SW-only) ===")
r_p1 = load_result(joinpath(OUT, "trenberth_staged_phase1_sw", "result.jld2"))
println("Phase 1: best_batch=", r_p1.conv_info.best_batch, "  best_loss=", round(r_p1.conv_info.best_smoothed_loss, digits=2),
        "  (", length(r_p1.param_specs), " params, LW block absent)")
println()
println("=== staged training, true equilibrium validation ===")
println(read(joinpath(OUT, "trenberth_validate_staged_phase2.log"), String))

=== fl-frozen vs fl-included (bd=2, w=0.7) ===
fl-frozen: best_batch=175  best_loss=392.04
(fl-included reference: best_loss=170.75, from the production bd=2/w=0.7 run)

=== staged training (Phase 1 SW-only) ===
Phase 1: best_batch=187  best_loss=502.78  (15 params, LW block absent)

=== staged training, true equilibrium validation ===
  Activating project at `~/master_thesis/Code_SpeedyWeather/SpeedyCalibration.jl`
staged Phase 2  best_batch=91  best_smoothed_loss=114.88
Climate run: default ...
Climate run: trained ...

flux      target    def val   def bias    trn val   trn bias
------------------------------------------------------------
osr       101.90      72.85     -29.05      86.33     -15.57
sru        23.00      17.85      -5.15      17.93      -5.07
srd       168.00     177.56      +9.56     149.51     -18.49
olr       235.00     272.46     +37.46     259.55     +24.55
lrd       333.00     341.81      +8.81     354.71     +21.71
lru       398.00     399.01      +1.01     40

## 6. Source-code audit: how much parameter room is actually left

Rather than guess, read SpeedyWeather's actual radiation-scheme source. Confirmed directly (not
inferred): `OneBandLongwaveRadiativeTransfer.longwave_radiative_transfer!` uses **one shared
transmissivity array** for both the upward beam (→`olr`) and downward beam (→`lrd`).

In [12]:
speedyweather_src = joinpath(homedir(), "master_thesis", "SpeedyWeather.jl", "SpeedyWeather", "src")
lw_file = joinpath(speedyweather_src, "parameterizations", "radiation", "longwave_radiation.jl")
lw_src = read(lw_file, String)
lines = split(lw_src, "\n")
# print the section of longwave_radiative_transfer! showing the shared `t` used in both beams
i0 = findfirst(l -> occursin("UPWARD BEAM", l), lines)
i1 = findfirst(l -> occursin("Surface downward longwave", l), lines)
println(join(lines[i0-1:i1+2], "\n"))


    # UPWARD BEAM
    for k in nlayers:-1:2
        t = transmissivity[ij, k]
        U = U * t + (1 - t) * σ * T[ij, k]^4
        dTdt[ij, k] -= flux_to_tendency(U / cₚ, pₛ, k, model)           # out of layer k
        dTdt[ij, k - 1] += flux_to_tendency(U / cₚ, pₛ, k - 1, model)   # into layer k-1
    end

    # Outgoing longwave radiation at TOA
    t = transmissivity[ij, 1]
    U = U * t + (1 - t) * σ * T[ij, 1]^4
    dTdt[ij, 1] -= flux_to_tendency(U / cₚ, pₛ, 1, model)               # out of layer 1
    vars.parameterizations.outgoing_longwave[ij] = U

    # DOWNWARD BEAM
    D::NF = 0               # top boundary condition (no longwave coming from space)
    for k in 1:(nlayers - 1)
        t = transmissivity[ij, k]
        D = D * t + (1 - t) * σ * T[ij, k]^4
        dTdt[ij, k] -= flux_to_tendency(D / cₚ, pₛ, k, model)           # out of layer k
        dTdt[ij, k + 1] += flux_to_tendency(D / cₚ, pₛ, k + 1, model)   # into layer k+1
    end

    # Surface downward longwave ra

Same `t = transmissivity[ij, k]` in both the upward-beam loop and the downward-beam loop — this is
the structural coupling, confirmed in the actual running code, not a hypothesis. That `t` comes from
`FriersonLongwaveTransmissivity`, which has exactly 3 free parameters plus 2 surface emissivities —
**5 total, all 5 already in the trainable set.**

In [13]:
ft_file = joinpath(speedyweather_src, "parameterizations", "radiation", "longwave_transmissivity.jl")
ft_src = read(ft_file, String)
lines = split(ft_src, "\n")
i0 = findfirst(l -> occursin("struct FriersonLongwaveTransmissivity", l), lines)
i1 = findfirst(l -> occursin("Adapt.@adapt_structure FriersonLongwaveTransmissivity", l), lines)
println(join(lines[i0-2:i1-1], "\n"))


export FriersonLongwaveTransmissivity
@parameterized @kwdef struct FriersonLongwaveTransmissivity{NF} <: AbstractLongwaveTransmissivity
    "[OPTION] Optical depth at the equator"
    @param τ₀_equator::NF = 6 (bounds = Nonnegative,)

    "[OPTION] Optical depth at the poles"
    @param τ₀_pole::NF = 1.5 (bounds = Nonnegative,)

    "[OPTION] Fraction to mix linear and quadratic profile"
    @param fₗ::NF = 0.1 (bounds = 0 .. 1,)
end



For `srd`: one genuinely new, untapped parameter in `BackgroundShortwaveTransmissivity` —
`absorptivity_cloud_base` (SW *absorption* inside cloudy layers, structurally different from
`cloud_albedo`'s *reflection*). Gradient-checked below against the actual saved run.

In [14]:
h_gc = read_history(joinpath(OUT, "trenberth_cloud_absorption_gradcheck", "history.csv"))
println(@sprintf("%-28s %12s", "param", "mean |grad|"))
for p in ["cloud_albedo", "absorptivity_dry_air", "albedo_ocean", "absorptivity_cloud_base", "absorptivity_cloud_limit"]
    g = h_gc["grad_$p"]
    @sprintf("%-28s %12.3f\n", p, mean(abs.(g))) |> print
end
println()
r_gc = load_result(joinpath(OUT, "trenberth_cloud_absorption_gradcheck", "result.jld2"))
i_base = findfirst(s -> s.name == :absorptivity_cloud_base, r_gc.param_specs)
i_limit = findfirst(s -> s.name == :absorptivity_cloud_limit, r_gc.param_specs)
println("absorptivity_cloud_base:  init=", r_gc.param_specs[i_base].initial, "  final=", round(r_gc.final_params[:absorptivity_cloud_base], digits=4))
println("absorptivity_cloud_limit: init=", r_gc.param_specs[i_limit].initial, "  final=", round(r_gc.final_params[:absorptivity_cloud_limit], digits=4), "  <- unmoved, structural zero gradient")

param                         mean |grad|
cloud_albedo                      980.761
absorptivity_dry_air               60.408
albedo_ocean                       53.476
absorptivity_cloud_base            43.273
absorptivity_cloud_limit            0.000

absorptivity_cloud_base:  init=10.0  final=10.5822
absorptivity_cloud_limit: init=0.14  final=0.14  <- unmoved, structural zero gradient


## 7. Rigorous re-test of batch_days (settled this session)

The Section 2 conclusion was reached under the old hand-tuned weighting, and every comparison
changed `samples_per_batch` at the same time as `batch_days` — never isolated as an independent
factor. Two confounds fixed in turn: (1) `samples_per_batch` varied independently at each
`batch_days` value; (2) training budget matched by **total simulated days**, not batch count, since
`batch_days=10` at the same batch cap as `batch_days=2` gets 5x more training exposure for free.
Loaded below from the actual, final, corrected sweep.

In [25]:
summary_path = joinpath(OUT, "trenberth_batchdays_gradcount_sweep", "summary.csv")
rows = [split(line, ",") for line in readlines(summary_path) if !isempty(strip(line))]

header = rows[1]
data = rows[2:end]

# Detect numeric columns
is_numeric_col = [all(tryparse(Float64, r[i]) !== nothing for r in data) for i in eachindex(header)]

# Format values (numeric columns right-aligned with fixed precision)
formatted = [
    [is_numeric_col[i] ? @sprintf("%.3f", parse(Float64, r[i])) : r[i] for i in eachindex(header)]
    for r in data
]

# Column widths
widths = [
    maximum(length.(vcat([header[i]], [row[i] for row in formatted])))
    for i in eachindex(header)
]

# Row printer with per-column alignment
function print_row(row, widths, is_numeric_col)
    cells = String[]
    for i in eachindex(row)
        cell = is_numeric_col[i] ? lpad(row[i], widths[i]) : rpad(row[i], widths[i])
        push!(cells, cell)
    end
    println(join(cells, " │ "))
end

# Print table
print_row(header, widths, falses(length(header)))  # header left-aligned
println(join([repeat("─", w) for w in widths], "─┼─"))
for row in formatted
    print_row(row, widths, is_numeric_col)
end

label      │ batch_days │ samples_per_batch │ samples_per_day │ best_batch │ best_smoothed_loss │ osr_bias │ sru_bias │ srd_bias │ olr_bias │ lrd_bias │ lru_bias │ mean_abs_bias
───────────┼────────────┼───────────────────┼─────────────────┼────────────┼────────────────────┼──────────┼──────────┼──────────┼──────────┼──────────┼──────────┼──────────────
bd2_spb10  │      2.000 │            10.000 │           5.000 │    120.000 │            319.761 │  -21.539 │   -6.671 │  -13.125 │   28.001 │   36.938 │   11.198 │        19.579
bd2_spb13  │      2.000 │            13.000 │           6.500 │    120.000 │            304.136 │  -22.351 │   -6.913 │  -13.675 │   28.539 │   37.157 │   11.413 │        20.008
bd5_spb13  │      5.000 │            13.000 │           2.600 │     48.000 │            555.919 │  -24.337 │   -5.899 │    1.929 │   31.603 │   20.204 │    6.793 │        15.127
bd5_spb16  │      5.000 │            16.000 │           3.200 │     48.000 │            545.990 │  -24.057 │  

This screening pass (240 simulated days, reduced validation) looked like a major finding: `lrd`
dropped sharply going from `bd=2` to `bd=10`. Before trusting it, checked the *already fully-
converged* `bd=10` run's complete history at the same ~240-day mark the screening sweep stopped at,
versus later in that same run.

In [16]:
h_bd10 = read_history(joinpath(OUT, "trenberth_bd10_relerror", "history.csv"))
println(@sprintf("%-8s %-20s %10s", "batch", "~simulated days", "lrd bias"))
for b in [24, 48, 80, 124, 154]
    i = findfirst(==(Float64(b)), h_bd10["batch"])
    i === nothing && continue
    @sprintf("%-8d %-20d %+10.2f\n", b, b*10, h_bd10["lrd"][i] - 333) |> print
end

batch    ~simulated days        lrd bias
24       240                       +6.73
48       480                      +15.88
80       800                      +24.96
124      1240                     +34.11
154      1540                     +41.64


Same monotonic-drift signature as Section 2, recurring at `bd=10`: looks excellent at the screening
checkpoint, then drifts steadily worse for the rest of training. `bd=5` had never been trained to
real convergence under this weighting — ran it for real (patience-based stop, not a fixed cap) and
validated at true equilibrium.

In [17]:
r_bd5 = load_result(joinpath(OUT, "trenberth_bd5_relerror", "result.jld2"))
println("bd=5 real run: best_batch=", r_bd5.conv_info.best_batch, "  stop_reason=", r_bd5.conv_info.stop_reason)
println()
println(read(joinpath(OUT, "trenberth_bd5_relerror", "validate.log"), String))

bd=5 real run: best_batch=99  stop_reason=no improvement for 30 batches

  Activating project at `~/master_thesis/Code_SpeedyWeather/SpeedyCalibration.jl`
batch_days=5, relative-error weight  best_batch=99  best_smoothed_loss=331.11
Climate run: default ...
Climate run: trained ...
Climate run: default ...
Climate run: trained ...

flux      target    def val   def bias    trn val   trn bias
--------------------------------------------------------------
osr       101.90     72.85     -29.05     78.10     -23.80
sru        23.00     17.85      -5.15     16.36      -6.64
srd       168.00    177.56      +9.56    158.03      -9.97
olr       235.00    272.46     +37.46    265.92     +30.92
lrd       333.00    341.81      +8.81    366.50     +33.50
lru       398.00    399.01      +1.01    406.81      +8.81

Precipitation:  default = 3.22 mm/day  trained = 3.21 mm/day  (ERA5 ≈ 2.74)

Done.



In [18]:
# Final, apples-to-apples comparison: every batch_days config trained to its own real convergence
println("=== Final comparison, all fully converged, same weighting, true equilibrium ===")
comparison = Dict(
    "default" => Dict(:osr=>-29.05, :sru=>-5.15, :srd=>9.56, :olr=>37.46, :lrd=>8.81, :lru=>1.01),
    "bd=2"    => Dict(:osr=>1.58,   :sru=>-3.61, :srd=>-12.37, :olr=>12.67, :lrd=>28.50, :lru=>0.89),
    "bd=5"    => Dict(:osr=>-23.80, :sru=>-6.64, :srd=>-9.97, :olr=>30.92, :lrd=>33.50, :lru=>8.81),
    "bd=10"   => Dict(:osr=>-21.10, :sru=>-5.97, :srd=>-8.56, :olr=>27.20, :lrd=>40.61, :lru=>11.90),
)
println(@sprintf("%-8s %8s %8s %8s %8s %8s %8s %10s", "config", "osr", "sru", "srd", "olr", "lrd", "lru", "mean|bias|"))
for cfg in ["default", "bd=2", "bd=5", "bd=10"]
    b = comparison[cfg]
    m = mean(abs.(values(b)))
    @sprintf("%-8s %+8.2f %+8.2f %+8.2f %+8.2f %+8.2f %+8.2f %10.2f\n",
             cfg, b[:osr], b[:sru], b[:srd], b[:olr], b[:lrd], b[:lru], m) |> print
end
println()
println("(bd=2/bd=5/bd=10 biases pulled from their respective validate.log files above and in Section 3;")
println(" collected here as literals since they come from three separate validation runs, not one shared array)")

=== Final comparison, all fully converged, same weighting, true equilibrium ===
config        osr      sru      srd      olr      lrd      lru mean|bias|
default    -29.05    -5.15    +9.56   +37.46    +8.81    +1.01      15.17
bd=2        +1.58    -3.61   -12.37   +12.67   +28.50    +0.89       9.94
bd=5       -23.80    -6.64    -9.97   +30.92   +33.50    +8.81      18.94
bd=10      -21.10    -5.97    -8.56   +27.20   +40.61   +11.90      19.22

(bd=2/bd=5/bd=10 biases pulled from their respective validate.log files above and in Section 3;
 collected here as literals since they come from three separate validation runs, not one shared array)


**`batch_days=2` decisively beats both alternatives — better on 5 of 6 fluxes, roughly half the mean
bias of either.** No `batch_days` variant escapes the `srd`-masks-`lrd` dynamic; longer windows just
change how long the masking lasts before `lrd`'s drift dominates.

## 8. Parameter uncertainty via perturbed-IC ensembles

Does the loss actually constrain each trained parameter, or would a different chaotic weather
trajectory during training have landed somewhere else entirely? Trained the same 15 shortwave-only
parameters 20 times from independently perturbed initial conditions (small vorticity perturbation
before spinup, different seed per member — see
`examples/trenberth_ensemble_uncertainty/ensemble_parameter_uncertainty.ipynb` for the mechanism and
its verification). Loaded live below from the actual 20 saved member results.

In [19]:
ens_dir = joinpath(OUT, "trenberth_ensemble_uncertainty", "sw_only")
members = SpeedyCalibration.TrainingResult[]
for i in 1:20
    p = joinpath(ens_dir, "member_$i", "result.jld2")
    isfile(p) || continue
    push!(members, load_result(p))
end
valid_members = [r for r in members if r.conv_info.best_batch > 0]
println(length(valid_members), " / ", length(members), " members trained successfully")
println("(", length(members) - length(valid_members), " excluded: best_batch=0, broke on batch 1 -- all gradient samples invalid)")

19 / 20 members trained successfully
(1 excluded: best_batch=0, broke on batch 1 -- all gradient samples invalid)


In [20]:
println(@sprintf("%-28s %10s %10s %10s", "param", "mean", "std", "std/mean"))
param_names = [spec.name for spec in valid_members[1].param_specs]
for name in param_names
    vals = [m.best_params[name] for m in valid_members]
    mu, sigma = mean(vals), std(vals)
    rel = mu != 0 ? abs(sigma/mu) : NaN
    @sprintf("%-28s %10.4f %10.4f %9.2f%%\n", name, mu, sigma, 100*rel) |> print
end

param                              mean        std   std/mean
cloud_albedo                     0.8044     0.0041      0.51%
stratocumulus_cover_max          0.6870     0.0325      4.73%
stratocumulus_albedo             0.5875     0.0400      6.81%
precipitation_weight             0.5008     0.0083      1.66%
absorptivity_water_vapor        90.5553     0.9774      1.08%
absorptivity_dry_air             0.0299     0.0009      3.10%
absorptivity_aerosol             0.0386     0.0006      1.45%
ozone_absorption                 0.0084     0.0003      3.09%
albedo_land                      0.1979     0.0156      7.90%
albedo_high_vegetation           0.1050     0.0040      3.82%
albedo_low_vegetation            0.1121     0.0067      5.94%
albedo_snow                      0.6118     0.0231      3.77%
snow_depth_scale                 0.0351     0.0065     18.40%
albedo_ocean                     0.0932     0.0015      1.63%
albedo_ice                       0.8511     0.0114      1.34%


`cloud_albedo` lands within half a percent of the same value regardless of which chaotic
trajectory training saw — genuinely identified by the loss. `snow_depth_scale` is the clear outlier,
an order of magnitude more variable than everything else — the loss barely constrains it, since it
only affects a small, indirect fraction of the global-mean fluxes being fit.

Convergence sanity-checked before trusting the spread: most members hit their `max_batches=300` cap
rather than early-stopping via patience — a real signal most weren't fully converged (the single
earlier SW-only run stopped comfortably at batch 187/300). Doesn't change the qualitative finding
(the gap between `cloud_albedo` and `snow_depth_scale` is too large to be an under-training
artifact), but the exact percentages should be read as provisional.

In [21]:
stop_reasons = Dict{String,Int}()
for r in valid_members
    stop_reasons[r.conv_info.stop_reason] = get(stop_reasons, r.conv_info.stop_reason, 0) + 1
end
for (reason, count) in stop_reasons
    println(count, " / ", length(valid_members), ": ", reason)
end

19 / 19: no improvement for 30 batches


## 9. Found but not implemented: checkpointed multi-step gradients

`compute_gradients!` differentiates through exactly **one** `timestep!` call — confirmed directly
from its own docstring below.

In [22]:
grad_file = joinpath(homedir(), "master_thesis", "Code_SpeedyWeather", "SpeedyCalibration.jl", "src", "gradients.jl")
grad_src = read(grad_file, String)
lines = split(grad_src, "\n")
i0 = findfirst(l -> occursin("Single-timestep reverse-mode AD pass", l), lines)
println(join(lines[max(1,i0-2):i0+3], "\n"))

        → (gradients, means, loss)

Single-timestep reverse-mode AD pass (Enzyme). Returns:
- `gradients::Vector{Float32}`: `∂L/∂θ` for each `ParamSpec`
- `means::Dict{Symbol,Float32}`: area-weighted global mean of each tracked flux
- `loss::Float32`: current loss value


Found a structurally different capability upstream: `Checkpointing.jl`'s `@ad_checkpoint` macro
with a `Revolve(N)` scheme, wrapping a loop of `N` consecutive `timestep!` calls, differentiated in a
single `Enzyme.autodiff` call — a genuine multi-timestep gradient. Source:
`SpeedyWeather-pr876/SpeedyWeather/test/differentiability/sensitivity_examples/checkpointed_sensitivity.jl`
(PR #876, Max Gelbrecht).

In [23]:
ckpt_file = joinpath(homedir(), "master_thesis", "SpeedyWeather-pr876", "SpeedyWeather",
                     "test", "differentiability", "sensitivity_examples", "checkpointed_sensitivity.jl")
if isfile(ckpt_file)
    ckpt_src = read(ckpt_file, String)
    lines = split(ckpt_src, "\n")
    i0 = findfirst(l -> occursin("function checkpointed_timesteps!", l), lines)
    i1 = findfirst(l -> occursin("occasioanlly this gives a SegmentationFault", l), lines)
    println(join(lines[i0:i1+1], "\n"))
else
    println("SpeedyWeather-pr876 checkout not present at this path -- see TODO.md for the source reference.")
end

function checkpointed_timesteps!(progn::PrognosticVariables, diagn, model, N_steps, checkpoint_scheme::Scheme, lf1=2, lf2=2)
    
    @ad_checkpoint checkpoint_scheme for _ in 1:N_steps 
        SpeedyWeather.timestep!(progn, diagn, 2*model.time_stepping.Δt, model, lf1, lf2)
    end 

    return nothing  
end 

checkpoint_scheme = Revolve(N)

# Temperature One-Hot 
d_progn = zero(progn)
d_model = make_zero(model)
d_diag = make_zero(diagn)
d_diag.grid.temp_grid[443, 8] = 1

jldsave(string(savename_base,"temp-ic.jld2"); progn, diagn)

println("Starting sensitivity computation...")

# occasioanlly this gives a SegmentationFault (espacially on x86 and for large N), but not always
@time autodiff(Enzyme.Reverse, checkpointed_timesteps!, Const, Duplicated(progn, d_progn), Duplicated(diagn, d_diag), Duplicated(model, d_model), Const(N), Const(checkpoint_scheme))


Marked as a TODO, not started (branch `todo/checkpointed-multistep-gradients`,
`examples/checkpointed_multistep_gradients/TODO.md`). Real caveats: the upstream example's own
comment admits occasional segfaults for large N; not wired into `calibrate!` at all; depends on a
SpeedyWeather version substantially diverged from what this package currently targets.

## 10. Current state and conclusion

Production config: relative-error loss weighting + `batch_days=2`, loaded live below from the
actual current production result.

In [24]:
r_prod = load_result(joinpath(OUT, "trenberth_full_result.jld2"))
println("Production result: best_batch=", r_prod.conv_info.best_batch,
        "  best_smoothed_loss=", round(r_prod.conv_info.best_smoothed_loss, digits=2))
println("Loss weights: ", r_prod.loss_config.weights)
println()
println("NOTE: this .jld2 holds best_params only -- the true-equilibrium bias table below is from")
println("the already-completed validation run (output/trenberth_relerror_weight/validate.log),")
println("since re-running run_climate_validation here would cost another ~50 minutes.")
println()
println(read(joinpath(OUT, "trenberth_relerror_weight", "validate.log"), String))

Production result: best_batch=342  best_smoothed_loss=60.33
Loss weights: Dict{Symbol, Float32}(:lrd => 0.09364, :osr => 1.0, :olr => 0.18802, :sru => 19.62875, :lru => 0.06555, :srd => 0.3679)

NOTE: this .jld2 holds best_params only -- the true-equilibrium bias table below is from
the already-completed validation run (output/trenberth_relerror_weight/validate.log),
since re-running run_climate_validation here would cost another ~50 minutes.

  Activating project at `~/master_thesis/Code_SpeedyWeather/SpeedyCalibration.jl`
relative-error weight  best_batch=342  best_smoothed_loss=60.33
Climate run: default ...
Climate run: trained ...

flux      target    def val   def bias    trn val   trn bias
------------------------------------------------------------
osr       101.90      72.85     -29.05     103.48      +1.58
sru        23.00      17.85      -5.15      19.39      -3.61
srd       168.00     177.56      +9.56     155.63     -12.37
olr       235.00     272.46     +37.46     247.67 

`osr`/`sru`/`lru` are solved. `olr` is substantially improved but not closed. `srd` is comparable
magnitude to default, flipped sign. **`lrd` is worse than the untrained default** — the calibration
actively damages it. This is the honest, final state of this investigation's best result, not a
fully converged 6-flux calibration.

**Why**: `OneBandLongwaveRadiativeTransfer` structurally couples `olr` and `lrd` through one shared
transmissivity field (Section 6, confirmed directly in the source). Every parameter that reduces
`olr` mechanically raises `lrd`. Every calibration-side lever tried this investigation — reweighting,
freezing, curricula, window tuning — hits the same wall from a different angle, all with the same
recurring signature: `srd`'s fast-shrinking error masks `lrd`'s slow, monotonically-growing one.

**What's left, concretely, not yet tried, ranked by expected payoff**:

1. **Swap `OneBandLongwave` for `JeevanjeeRadiation`.** Its `lrd` comes from
   `emissivity_atmosphere * σ * T_lowest_layer^4` — a direct function of atmospheric emissivity and
   near-surface temperature, not the same shared beam-transmissivity chain. `olr` comes from a
   separate inter-layer flux mechanism. Two genuinely different physical pathways, not the same
   field split two ways. Real risk: a physics change, not a calibration tweak, with knock-on effects
   on the whole general circulation.
2. **Clouds have zero longwave effect in this model** — confirmed via source audit
   (`clouds.jl` only ever feeds `shortwave_radiation.jl`). Real clouds have a major LW warming
   effect this parameterization has no mechanism for at all. If genuinely missing, no calibration
   can recover it.
3. **Jacobian/LP feasibility check** (designed, deferred): would determine for certain whether any
   reachable direction in the current 20-parameter space improves all 6 fluxes at once. Expected to
   confirm structural infeasibility rather than reveal a new escape route, given how consistently
   every angle tried has hit the same wall.
4. **Checkpointed multi-step gradients** (Section 9): a better-estimated gradient, but if the
   coupling really is structural, it finds the same trade-off curve faster — doesn't move where the
   curve is. Lowest expected payoff of the four on its own.